# IMPORTS

First we started with some python imports. For model building we used the <mark>**segmentation_models_pytorch library**</mark>. This has library has a lot segmentation model architechtures. Before you can import the library it needs to be ensured that the library is installed in the conda environment. For this in the anaconda terminal just type the below command to install the library:-

<div class="alert alert-block alert-success">
<b>Conda Command:</b> conda install -q segmentation-models-pytorch
</div>

Once, the libray is installed it can be imported in the script safely. We also imported the utility script Trainer for model training.

In [ ]:
import pandas as pd
import numpy as np
import segmentation_models_pytorch as smp
from Utilities.Trainer import Trainer
import torch
import torch.nn as nn
from torchvision import models
import torch.multiprocessing as mp
import matplotlib.pyplot as plt
%matplotlib inline

The below statement helps in better error tracking when training is done in cuda. More details about this can be found in the below url:-

https://pytorch.org/docs/stable/notes/cuda.html

In [ ]:
CUDA_LAUNCH_BLOCKING="1"

# Model Building

Next a FPN model with Resnet34 encoder was created with imagenet weights from the smp library.

In [ ]:
torch.manual_seed(42)
ENCODER = 'resnet34'
ENCODER_WEIGHTS = 'imagenet'
CLASSES = 6
ACTIVATION = None 

# create segmentation model with pretrained encoder
model = smp.FPN(
    encoder_name=ENCODER, 
    encoder_weights=ENCODER_WEIGHTS, 
    in_channels=3,
    classes=CLASSES, 
    activation=ACTIVATION,
)
print(model)

# Model Training

Next the model training was started with 20 epochs and initial learning rate as 5e-04. The model checkpoint was saved whenever the validation dice was reported the greater than the previous best validation dice.

In the below results you can see the training starting from epoch 6. This is because the kernel was initally interuptted after epoch 5. So, I had to start all over again. Since, checkpoint was saved upto epoch number 5 so it started from epoch 6.

In [ ]:
lr = 5e-04
epochs = 20
path = "./Models/model_fpn_resnet.pth"

In [ ]:
model_trainer = Trainer(model, lr, epochs, path)
print(f"Using device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
print(f"Total memory allocated in GB: {round(torch.cuda.memory_allocated(0)/1024**3,1)}")
print(f"Total memory cached in GB: {round(torch.cuda.memory_reserved(0)/1024**3,1)}")
model_trainer.start()

After the model training was done the loss, dice_scores and iou_scores were plotted. 

In [ ]:
losses = model_trainer.losses
dice_scores = model_trainer.dice_scores 
iou_scores = model_trainer.iou_scores

def plot(scores, name):
    plt.figure(figsize=(15,5))
    plt.plot(range(len(scores["train"])), scores["train"], label=f'train {name}')
    plt.plot(range(len(scores["train"])), scores["val"], label=f'val {name}')
    plt.title(f'{name} plot'); plt.xlabel('Epoch'); plt.ylabel(f'{name}');
    plt.legend(); 
    plt.show()

plot(losses, "BCE loss")
plot(dice_scores, "Dice score")
plot(iou_scores, "IoU score")

The above results the overall training and validation dice and IoU goes up with epochs and training and validation loss goes down with epochs.

# Convert to ONNX

Lastly the trained model is converted to ONNX format using the Utility script ONNX_Converter.py.

In [ ]:
! python ./Utilities/ONNX_converter.py FPN resnet34  imagenet  6 ./Models/model_fpn_resnet.pth ./ONNX_models/fpn_resnet.onnx